In [3]:
%pip install datatrove
%pip install tokenizer

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


step: tokenizer -> shadding -> embedding

tokenizer part

In [6]:
def batch_iterator(batch_size=1000, max_docs=100_000):
        batch = []
        doc_count = 0

        for ex in sub_dataset:
            batch.append(ex["text"])
            doc_count += 1
            if len(batch) == batch_size:
                yield batch
                batch = []
            if doc_count >= max_docs:
                break
        if batch:
            yield batch

In [7]:
import os
from datasets import load_dataset
from tokenizers import Tokenizer, AddedToken
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel, Digits, Sequence
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer

LEN_OF_SUB_DATASET = 1_000_000

if __name__ == "__main__":
    print("huggingface fineweb edu 10 bt 중에서 일부 로드 하기")
    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True
    )

    sub_dataset = dataset.take(LEN_OF_SUB_DATASET)

    #tokenizer setting 과정
    tokenizer = Tokenizer(BPE())
    tokenizer.pre_tokenizer = Sequence(
        [ByteLevel(add_prefix_space=False), #Byte 단위로 text 변환, 
         Digits(individual_digits=True)
        ])
    tokenizer.post_processor = ByteLevelProcessor(tirm_offsets=False)
    tokenizer.decoder = ByteLevelDecoder()

    special_tokens = [AddedToken(token, special=True, normalized=True) 
                      for token in ["[PAD]", "[BOS]", "[EOS]"]]

    special_tokens += [AddedToken(f"[UNUSED]_{i}", special=True, normalized=False) for i in range(10)]
    #UNUSED 저거에서 10은 예비토큰수임: 

    #token 학습하기
    trainer = BpeTrainer(
        vocab_size=20000,
        initial_alphabet=ByteLevel.alphabet(),
        max_token_length=10,
        special_tokens=special_tokens,
    )


    print("BPE 토크나이저 ")
    tokenizer.train_from_iterator(batch_iterator(batch_size=1000, max_docs=100_000), trainer=trainer)

    model_path = "models/tokenizer.json"
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    tokenizer.save(model_path)

    print("tokenizer 끝")

c:\Users\kimji\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


huggingface fineweb edu 10 bt 중에서 일부 로드 하기


BPE 토크나이저 
tokenizer 끝


shadding part

In [8]:
from datatrove.executor import LocalPipelineExecutor
from datatrove.pipeline.readers import HuggingFaceDatasetReader
from datatrove.pipeline.base import PipelineStep

import numpy as np

LIMITS = 10_000

class CustomTokenizerStep(PipelineStep):
    def __init__(self, tokenizer_path: str, output_dir: str, chunk_size:int = 50_000):
        super().__init__()
        self.tokenizer_path = tokenizer_path
        self.output_dir = output_dir
        self.chunk_size = chunk_size
        os.makedirs(output_dir, exist_ok=True)

    def run(self, data, rank: int = 0, word_size: int = 1):
        #word_size 쟤는 worker 의 수와 같다보 보면 됨
        tokenizer = Tokenizer.from_file(self.tokenizer_path)

        all_tokens = []
        shard_counter = 0

        print(f"[Task {rank}] 토큰화 시작, 바이너리 샤딩")

        for doc in data:
            text = doc.text # 문서 text 가져옴
            if not text: # 근데 없으면 건너뜀
                continue

            
            token_ids = tokenizer.encode(text).ids
            token_ids.append(tokenizer.token_to_id("[EOS]")) # 문장 끝에 대해서 대비하기

            all_tokens.extend(token_ids) # 모든 token id 저장

            if len(all_tokens) >= self.chunk_size:
                arr = np.array(all_tokens, dtype=np.uint)
                shard_path = os.path.join(self.output_dir, f"shard_{rank}_{shard_counter:05d}.bin")
                arr.tofile(shard_path)

                del arr
                all_tokens = []
                shard_counter += 1

                

        if all_tokens:
            arr = np.array(all_tokens, dtype=np.uint32)
            shard_path = os.path.join(self.output_dir, f"shard_{rank}_{shard_counter:05d}.bin")
            arr.tofile(shard_path)
            
        print("샤딩 끝!!")
#저 if 문 부터 저게 뭐하는 애인지 설명하자면 모든 토큰은 한 청크로 잘라서 각 청크마다 다음과 같은 작업을 함
#그 후로는 샤딩한 경로를 따라 파일로 저장

In [9]:
if __name__=="__main__":

    os.environ["HF_DATASETS_CACHE"] = "./data/hf_cache"
    os.makedirs("./data/hf_cache", exist_ok=True)

    output_dir = "data/bin_shards"
    os.makedirs("./data/bin_shards", exist_ok=True)

    reader = HuggingFaceDatasetReader(
        dataset="HuggingFaceFW/fineweb-edu",
        dataset_options={"name": "sample-10BT", "split": "train"},
        text_key="text",
        streaming=True,
        limit=LIMITS
    )

    tokenizer_step = CustomTokenizerStep(
        tokenizer_path="models/tokenizer.json",
        output_dir=output_dir
    )

    executor = LocalPipelineExecutor(
        pipeline=[
            reader,
            tokenizer_step
        ],
        tasks=1,
        logging_dir="logs/datatrove"
    )

    print("샤딩 실행")
    executor.run()
    print("샤딩 끝")

2026-11-09 04:36:27.968 | INFO     | datatrove.executor.local:run:109 - Not doing anything as all 1 tasks have already been completed.


샤딩 실행
샤딩 끝


embedding part

In [10]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class MemmapDataset(Dataset):
    def __init__(self, shard_dir, seq_len=128):
        self.seq_len = seq_len
        shard_files = sorted(glob.glob(os.path.join(shard_dir, "*.bin")))

        if not shard_files:
            raise FileNotFoundError(f"{shard_dir}경로에 binard shard 가 없음")

        self.mmap_data_list = []
        self.total_tokens = 0

        for fpath in shard_files:
            arr = np.memmap(fpath, dtype=np.uint32, mode='r')
            self.mmap_data_list.append(arr)
            self.total_tokens += len(arr)

        print(f"{len(shard_files)}샤드 로드됨, 총 토큰 수: {self.total_tokens}")

    def __len__(self):
        if self.total_tokens <= self.seq_len:
            return 0
        return (self.total_tokens - self.seq_len - 1) // self.seq_len + 1

    def __getitem__(self, idx):
        start_idx = idx * self.seq_len
        end_idx = start_idx + self.seq_len + 1
        chunk = self._get_tokens(start_idx, end_idx)

        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)

        return x, y

    def _get_tokens(self, start, end):
        current_pos = 0
        result = []

        for mmap_arr in self.mmap_data_list:
            arr_len = len(mmap_arr)
            if (current_pos + arr_len > start) and (current_pos < end):
                local_start = max(0, start - current_pos)
                local_end = min(arr_len, end - current_pos)
                result.append(mmap_arr[local_start:local_end])

            current_pos += arr_len

            if current_pos >= end:
                break
        if not result:
            raise IndexError(f"Requested token range [{start}, {end}) is out of bounds for the available shards.")
        return np.concatenate(result)

In [11]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, n_embed, block_size, dropout=0.1, pad_id=0):
        super().__init__()
        self.token = nn.Embedding(vocab_size, n_embed, padding_idx=pad_id)
        self.position = nn.Embedding(block_size, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, ids):
        b, t = ids.shape
        positions = torch.arange(t, device=ids.device)
        return self.dropout(self.token(ids) + self.position(positions))

In [13]:
if __name__=="__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("그냥 임베딩")

    vocab_size = 20_000 # 추후에 입력 파일 제공 예정
    block_size = 128
    n_embed = 256

    dataset = MemmapDataset(shard_dir="data/bin_shards", seq_len=block_size)
    data_loader = DataLoader(dataset, batch_size=4, shuffle=True)

    embedding_layer = TokenPositionEmbedding(vocab_size, n_embed, block_size).to(device)
    optimizer = optim.AdamW(embedding_layer.parameters(), lr=1e-4)

    print("시작")

    output_cnt = 0
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)

        embedded_output = embedding_layer(x)
        print(f"입력 shape: {x.shape}")
        print(f"결과 shape: {embedded_output.shape}")

        output_cnt += 1
        if output_cnt > 100:
            break

    print("임베딩 정상(?)종료")

그냥 임베딩
223샤드 로드됨, 총 토큰 수: 23395614
시작
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4, 128])
결과 shape: torch.Size([4, 128, 256])
입력 shape: torch.Size([4,